# 01 — Fixed business features and temporal inputs

The four existing filenames are retained. This revision replaces the previous feature-ranking experiment with the fixed V63 business feature list. The folder name is retained for compatibility; no feature selection runs here. Do not regenerate these notebooks with the older experiment build script or use the old folder README as this revision's run guide.

**Source blocker:** the repository contains snapshot values/readers, but no verified V63 historical feature-generation SQL or patient observation-coverage rule. The dictionary explains meaning, not every calculation. Notebook 01 displays the audit and stops until the missing calculations and coverage are supplied in its reconstruction cell. No historical values, coverage percentages or performance are claimed in this delivery.

Existing population: 23,151 patient snapshots, 12,447 patients, 1,345 positives; PATIENT_ID + END_DT; labels copied unchanged from the frozen source. The model-type FEATURES array supplies all 49 predictors in stored order. Calendar position 0 is newest; monthly positions 0–11 cover the original calendar buckets, with the current month truncated at END_DT. Quarterly positions 0–3 group exactly those buckets into consecutive three-month periods, not calendar-year quarters. Historical rolling windows can require source records earlier than this displayed 12-bucket sequence.

Architecture and optimizer remain the original temporal Transformer (128 width, 4 heads, 2 layers, FF256, dropout .2; weighted BCE, AdamW, validation-AP checkpointing). Mixed business values use the existing 49-feature model's TRAIN-only median/mean/std preprocessing principle rather than applying the old raw-count log1p. Padded rows are excluded from fitted statistics and re-zeroed afterwards. No feature is silently imputed to resolve missing reconstruction logic.

Only aggregate reports, tensors and model checkpoints are saved in the existing private warehouse pattern. No custom CSV/download export is provided. Clear all outputs before committing executed notebooks. TEST has previously been inspected and remains a retrospective check. Monthly/quarterly and masking are hypotheses; this two-model comparison alone does not isolate a causal masking effect from representation/preprocessing changes.


In [ ]:
# Connection and unchanged experimental settings
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply the existing private sf_options connection on the approved Spark runtime.')
sf_options_dl_poc = dict(sf_options)
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'H001'
RUN_ID = 'M001'
import re
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use short uppercase identifiers; use matching IDs in all four notebooks.')
# New output names prevent collision with prior saved models; original inputs remain unchanged.
EXPERIMENT_PREFIX = PREFIX + '_BUSINESS49_TEMPORAL_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
REFERENCE_MODEL_TABLE = PREFIX + '_MODEL_RUN_001'
REFERENCE_NAMES = {'checkpoint.pt', 'training_summary.json', 'training_history.csv', 'training_history.png'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=.0001, grad_clip=1., device='auto')
print('Fixed V63 business features; source cohort and RESP retained. Run MONTHLY then QUARTERLY.')

IMPLEMENTATION_SHA256 = 'eba7968a1cfa16f62857d5a220a5fda1a75fcbb3bb8faebc01987abcfb7978f1'


In [ ]:
# Embedded validation and warehouse helpers
"""Embedded notebook helpers: fixed business features, calendar grids and padding."""
import io
import json
import hashlib
import re
import marshal
import numpy as np
import pandas as pd


def require(condition, message):
    if not condition:
        raise ValueError(message)


def ordered_features():
    features, comparison = configured_features()
    require(len(features) == 49, 'V63 MODEL_TYPE.FEATURES must contain exactly 49 predictors.')
    return features, comparison


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def sequence_grid(metadata, representation):
    require(representation in ('MONTHLY', 'QUARTERLY'), 'Unknown representation.')
    width = 1 if representation == 'MONTHLY' else 3
    rows = []
    for r in metadata.itertuples():
        cutoff = pd.Timestamp(r.END_DT)
        month = cutoff.to_period('M')
        for step in range(12 // width):
            start = (month - width * step - (width - 1)).start_time.normalize()
            end = min(cutoff, (month - width * step).end_time.normalize())
            rows.append((r.PATIENT_ID, r.END_DT, step, start.strftime('%Y-%m-%d'),
                         end.strftime('%Y-%m-%d'), int(r.RESP)))
    grid = pd.DataFrame(rows, columns=['PATIENT_ID', 'END_DT', 'TIME_STEP', 'PERIOD_START', 'PERIOD_END', 'RESP'])
    require(not grid.duplicated(['PATIENT_ID', 'END_DT', 'TIME_STEP']).any(), 'Duplicate sequence keys.')
    require(grid.PERIOD_END.le(grid.END_DT).all(), 'Future period boundary.')
    return grid


def verify_periods(monthly, quarterly):
    keys = ['PATIENT_ID', 'END_DT']
    for frame, count in ((monthly, 12), (quarterly, 4)):
        require(not frame.duplicated(keys + ['TIME_STEP']).any(), 'Duplicate sequence row.')
        require(frame.groupby(keys).TIME_STEP.apply(lambda x: sorted(x) == list(range(count))).all(),
                'Missing or invalid sequence positions.')
    for q in range(4):
        m = monthly.loc[monthly.TIME_STEP.between(q * 3, q * 3 + 2)]
        bounds = m.groupby(keys).agg(PERIOD_START=('PERIOD_START', 'min'), PERIOD_END=('PERIOD_END', 'max'))
        actual = quarterly.loc[quarterly.TIME_STEP.eq(q)].set_index(keys)[['PERIOD_START', 'PERIOD_END']].sort_index()
        require(bounds.sort_index().equals(actual), 'Monthly and quarterly calendar periods differ.')


def reconcile_dictionary(features, dictionary):
    rows, assigned = [], set()
    for order, name in enumerate(features):
        exact = [r for r in dictionary if r['name_complete'] and r['name'] == name]
        candidates = exact or [r for r in dictionary if not r['name_complete'] and name.startswith(r['name'])]
        match = candidates[0] if len(candidates) == 1 else None
        if match:
            require(match['seq'] not in assigned, 'Dictionary entry matched multiple model features; confirm full names.')
            assigned.add(match['seq'])
        rows.append({'FEATURE_ORDER': order, 'FEATURE_NAME': name,
                     'DICTIONARY_SEQ': match['seq'] if match else None,
                     'MATCH': ('EXACT_NAME' if exact else 'UNIQUE_VISIBLE_PREFIX') if match else 'UNRESOLVED',
                     'BUSINESS_DEFINITION': match['definition'] if match else 'No unambiguous dictionary match.',
                     'FEATURE_TYPE': (match['type_label'] + ' (dictionary label; not a casting rule)') if match else 'UNRESOLVED',
                     'DEFAULT_STATUS': match['default_status'] if match else 'UNRESOLVED',
                     'NOTES': ((match.get('notes', '') + ('; clipped name matched by unique visible prefix' if not exact else ''))
                               if match else 'Confirm full feature name/definition against V63.')})
    unmatched = pd.DataFrame([r for r in dictionary if r['seq'] not in assigned])
    return pd.DataFrame(rows), unmatched


def reconstruction_audit(features, dictionary, rules):
    matched, extra = reconcile_dictionary(features, dictionary)
    require(not set(rules).difference(features), 'Historical rules include non-V63 features.')
    rows = []
    for row in matched.to_dict('records'):
        name = row['FEATURE_NAME']
        rule = rules.get(name)
        row['SOURCE/CALCULATION'] = SOURCE_PREFIX + '_MODEL_DATA; snapshot values only verified locally'
        row['HISTORICAL_RECONSTRUCTION_STATUS'] = row.pop('DEFAULT_STATUS')
        row['HISTORICAL_LOGIC'] = 'Historical V63 SQL and observation coverage not supplied; no replication or zero substitution.'
        if rule:
            status = rule.get('status')
            require(status in ('EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'), 'Invalid reconstruction status.')
            row['HISTORICAL_RECONSTRUCTION_STATUS'] = status
            row['SOURCE/CALCULATION'] = rule.get('source', '')
            row['HISTORICAL_LOGIC'] = rule.get('logic', '')
            row['NOTES'] += '; ' + rule.get('notes', '')
            if status in ('EXACT', 'APPROXIMATED'):
                require(all(rule.get(k) for k in ('source', 'logic', 'evidence', 'observation_logic')),
                        name + ': source, calculation evidence and observation logic are required.')
                require(callable(rule.get('builder')), name + ': executable historical calculation is missing.')
                if status == 'APPROXIMATED':
                    require(bool(rule.get('approximation')), name + ': describe the approximation explicitly.')
        rows.append(row)
    audit = pd.DataFrame(rows)
    counts = audit.HISTORICAL_RECONSTRUCTION_STATUS.value_counts().reindex(
        ['EXACT', 'APPROXIMATED', 'SNAPSHOT_ONLY', 'UNRESOLVED'], fill_value=0)
    require(len(audit) == int(counts.sum()) == 49, 'Reconstruction counts must sum to 49.')
    return audit, counts, extra


def require_reconstruction(audit, rules):
    blocked = audit.loc[~audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']), 'FEATURE_NAME'].tolist()
    require(not blocked, 'Historical reconstruction blocked. Supply verified V63 calculations and coverage for: ' + ', '.join(blocked))
    require(set(rules) == set(audit.FEATURE_NAME), 'Exactly 49 historical rules are required.')


def construct_sequence(grid, features, audit, rules, representation):
    require_reconstruction(audit, rules)
    keys = ['PATIENT_ID', 'END_DT', 'TIME_STEP']
    # Builders receive dates and identifiers only, never RESP or split membership.
    request = grid.drop(columns='RESP').copy()
    values, observed = [], []
    provenance = []
    for feature in features:
        rule = rules[feature]
        result = rule['builder'](request.copy(), representation)
        needed = keys + ['VALUE', 'IS_OBSERVED', 'MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE',
                         'OBSERVATION_EVIDENCE', 'PROVENANCE_KIND', 'PROVENANCE_NOTE']
        require(isinstance(result, pd.DataFrame) and set(needed).issubset(result.columns), feature + ': incomplete historical output.')
        result = result[needed].copy()
        require(not result[keys].isna().any().any() and not result.duplicated(keys).any(), feature + ': invalid historical keys.')
        require(len(result) == len(grid), feature + ': historical output must cover every requested position explicitly.')
        aligned = request.merge(result, on=keys, how='left', validate='one_to_one', indicator=True)
        require(aligned._merge.eq('both').all(), feature + ': missing historical keys.')
        require(aligned.IS_OBSERVED.isin([0, 1, False, True]).all(), feature + ': observation status is required for every position.')
        known = aligned.IS_OBSERVED.astype(bool).to_numpy()
        require(aligned.OBSERVATION_EVIDENCE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(),
                feature + ': availability must be evidenced, including unavailable periods.')
        for field in ('MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE'):
            dates = pd.to_datetime(aligned[field], errors='raise')
            require(dates.dt.tz is None and dates.dropna().eq(dates.dropna().dt.normalize()).all(),
                    feature + ': provenance dates must be exact dates; timestamp rules require explicit review.')
            require((dates.isna() | dates.le(pd.to_datetime(aligned.PERIOD_END))).all(), feature + ': future information detected in ' + field)
        numbers = pd.to_numeric(aligned.VALUE, errors='raise').to_numpy(dtype=np.float64)
        require(np.isfinite(numbers[known]).all(), feature + ': observed values must be finite; explicitly calculate observed zeros.')
        require(np.isnan(numbers[~known]).all(), feature + ': unavailable feature history must remain null before padding.')
        kinds = aligned.PROVENANCE_KIND
        require(kinds.isin(['EVENT_DERIVED', 'OBSERVED_EMPTY', 'STATIC', 'UNAVAILABLE']).all(), feature + ': invalid provenance kind.')
        require(aligned.PROVENANCE_NOTE.map(lambda v: isinstance(v, str) and bool(v.strip())).all(), feature + ': missing provenance explanation.')
        require(np.array_equal(kinds.ne('UNAVAILABLE').to_numpy(), known), feature + ': provenance and availability disagree.')
        event_rows = kinds.eq('EVENT_DERIVED')
        require(aligned.loc[event_rows, ['MAX_EVENT_DATE', 'MAX_AVAILABLE_DATE']].notna().all().all(),
                feature + ': event-derived values need both event and availability date maxima.')
        require(np.all(numbers[kinds.eq('OBSERVED_EMPTY')] == 0), feature + ': observed-empty provenance requires a defined zero value.')
        if kinds.eq('STATIC').any():
            require(rule.get('static_feature') is True and bool(rule.get('static_rationale')),
                    feature + ': static provenance requires a verified static-feature rule and rationale.')
        values.append(numbers)
        observed.append(known)
        provenance.append({'feature': feature, 'rule': {k: v for k, v in rule.items() if k != 'builder'},
                           'builder_sha256': hashlib.sha256(marshal.dumps(rule['builder'].__code__)).hexdigest(),
                           'observed_positions': int(known.sum())})
    raw = np.column_stack(values)
    known = np.column_stack(observed)
    # A token is fully observed only when all fixed 49 inputs are supported at this cutoff.
    # Partial feature availability is reported, not disguised as no activity.
    valid = known.all(axis=1)
    long = grid.copy()
    long[features] = raw
    long['AVAILABLE_FEATURE_COUNT'] = known.sum(axis=1)
    long['IS_VALID_TIMESTEP'] = valid.astype('int64')
    long['IS_PADDED'] = (~valid).astype('int64')
    long['PADDING_REASON'] = np.where(valid, '', 'Insufficient evidenced history for one or more fixed features')
    # Preserve raw missingness for review; only the model tensor receives padding zeros.
    n_steps = 12 if representation == 'MONTHLY' else 4
    X = np.where(valid[:, None], raw, 0).astype(np.float32).reshape(-1, n_steps, 49)
    mask = valid.reshape(-1, n_steps)
    require(np.isfinite(X).all(), 'NaN/Inf after padding.')
    require(np.array_equal(mask, long.IS_VALID_TIMESTEP.to_numpy().reshape(mask.shape)), 'Mask alignment failure.')
    observed_zero = known.all(axis=1) & np.all(raw == 0, axis=1)
    require(valid[observed_zero].all(), 'Observed zero activity was incorrectly masked.')
    return {'X': X, 'valid': mask, 'long': long, 'known': known, 'provenance': provenance,
            'representation': representation}


def sparsity_report(bundle, features):
    raw = bundle['long'][features].to_numpy(dtype=float)
    known = bundle['known']
    valid = bundle['valid']
    counts = known.sum(axis=0)
    zeros = ((raw == 0) & known).sum(axis=0)
    feature = pd.DataFrame({'FEATURE_NAME': features, 'OBSERVED_VALUES': counts,
                            'UNAVAILABLE_VALUES': (~known).sum(axis=0), 'OBSERVED_ZERO_VALUES': zeros,
                            'OBSERVED_ZERO_PERCENT': np.divide(100. * zeros, counts, out=np.full(49, np.nan), where=counts > 0)})
    eligible = int(known.sum())
    report = {'MODEL': bundle['representation'], 'TOTAL_TIMESTEPS': int(valid.size),
              'VALID_TIMESTEPS': int(valid.sum()), 'PADDED_TIMESTEPS': int((~valid).sum()),
              'VALID_PERCENT': float(valid.mean() * 100), 'PADDED_PERCENT': float((~valid).mean() * 100),
              'ALL_PADDED_SNAPSHOTS': int((~valid.any(axis=1)).sum()),
              'OBSERVED_FEATURE_ZERO_PERCENT': float(((raw == 0) & known).sum() * 100 / eligible) if eligible else None,
              'TENSOR_ZERO_PERCENT_INCLUDING_PADDING': float((bundle['X'] == 0).mean() * 100),
              'OBSERVED_ZERO_TIMESTEPS': int((valid.reshape(-1) & np.all(raw == 0, axis=1)).sum())}
    distribution = pd.Series(valid.sum(axis=1)).value_counts().sort_index().rename_axis('VALID_TIMESTEPS').reset_index(name='SNAPSHOTS')
    return report, feature, distribution


def display_sequence(bundle, features):
    frame = bundle['long'].copy()
    if bundle['representation'] == 'MONTHLY':
        frame = frame.rename(columns={'PERIOD_START': 'MONTH_START', 'PERIOD_END': 'MONTH_END'})
    else:
        frame = frame.rename(columns={'TIME_STEP': 'QUARTER_TIME_STEP', 'PERIOD_START': 'QUARTER_START', 'PERIOD_END': 'QUARTER_END'})
    print(bundle['representation'], 'raw historical values; unavailable values are null here and zero-padded only in the model tensor')
    for label in (0, 1):
        sample = frame.loc[frame.RESP.eq(label)].head(24)
        if not sample.empty:
            print('RESP =', label)
            display(sample)
    for state in (1, 0):
        sample = frame.loc[frame.IS_VALID_TIMESTEP.eq(state)].head(12)
        print('Valid' if state else 'Padded', 'positions:', 'available' if not sample.empty else 'none in this dataset')
        if not sample.empty:
            display(sample)
    raw = bundle['long']
    summary = raw.groupby(['PATIENT_ID', 'END_DT'], sort=True).agg(
        VALID=('IS_VALID_TIMESTEP', 'sum'), PADDED=('IS_PADDED', 'sum')).reset_index()
    activity = raw[features].fillna(0).ne(0).sum(axis=1)
    summary['NONZERO_VALUES'] = activity.groupby([raw.PATIENT_ID, raw.END_DT]).sum().to_numpy()
    choices = [('relatively dense', summary.sort_values(['VALID', 'NONZERO_VALUES'], ascending=False).head(1)),
               ('partially sparse', summary.loc[summary.VALID.gt(0)].sort_values('NONZERO_VALUES').head(1)),
               ('substantial padding', summary.loc[summary.PADDED.gt(0)].sort_values('PADDED', ascending=False).head(1))]
    for title, example in choices:
        print('Structural example:', title, '(selected without model scores)')
        if example.empty:
            print('No matching example in this dataset.')
        else:
            r = example.iloc[0]
            display(frame.loc[frame.PATIENT_ID.eq(r.PATIENT_ID) & frame.END_DT.eq(r.END_DT)])


def prepared_blobs(metadata, features, bundles, audit, comparison, snapshot_X):
    report = {'schema': 2, 'dataset_id': DATASET_ID, 'features': features,
              'population_sha256': digest_json(snapshot_records(metadata)),
              'snapshot_feature_sha256': array_hash(snapshot_X),
              'configuration_audit': comparison, 'audit': audit.astype(object).where(pd.notna(audit), None).to_dict('records'),
              'orientation': '0=newest; calendar buckets; most recent bucket truncated at END_DT',
              'implementation_sha256': IMPLEMENTATION_SHA256,
              'representations': {}}
    artifacts = {'population.json': canonical_json(snapshot_records(metadata)).encode()}
    for name, b in bundles.items():
        buffer = io.BytesIO()
        np.savez_compressed(buffer, X=b['X'], valid=b['valid'])
        artifacts[name + '.npz'] = buffer.getvalue()
        sparsity, _, distribution = sparsity_report(b, features)
        report['representations'][name] = {'shape': list(b['X'].shape), 'X_sha256': array_hash(b['X']),
              'valid_sha256': array_hash(b['valid']), 'sparsity': sparsity,
              'valid_distribution': distribution.to_dict('records'), 'provenance': b['provenance']}
    artifacts['manifest.json'] = canonical_json(report).encode()
    return artifacts


PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}


def load_prepared():
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs['manifest.json'])
    require(manifest['schema'] == 2 and manifest['dataset_id'] == DATASET_ID, 'Prepared dataset ID/version changed.')
    require(manifest['implementation_sha256'] == IMPLEMENTATION_SHA256, 'Prepared dataset was produced by different code.')
    features, _ = ordered_features()
    require(features == manifest['features'], 'Authoritative feature order changed after preparation.')
    metadata = pd.DataFrame(json.loads(blobs['population.json']), columns=['PATIENT_ID', 'END_DT', 'RESP'])
    metadata = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    population_check(metadata)
    source = normalize_metadata(read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas()).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(metadata.equals(source), 'Frozen population/RESP changed after preparation.')
    require(digest_json(snapshot_records(metadata)) == manifest['population_sha256'], 'Population hash mismatch.')
    audit = pd.DataFrame(manifest['audit'])
    require(audit.FEATURE_NAME.tolist() == features and audit.HISTORICAL_RECONSTRUCTION_STATUS.isin(['EXACT', 'APPROXIMATED']).all(), 'Unsupported or misordered reconstruction audit.')
    bundles = {}
    for name, count in (('MONTHLY', 12), ('QUARTERLY', 4)):
        with np.load(io.BytesIO(blobs[name + '.npz']), allow_pickle=False) as arrays:
            X, valid = arrays['X'].copy(), arrays['valid'].copy()
        require(X.shape == (len(metadata), count, 49) and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid tensor or mask dimensions.')
        require(np.isfinite(X).all() and np.all(X[~valid] == 0), 'Invalid padding or nonfinite input.')
        require(array_hash(X) == manifest['representations'][name]['X_sha256'] and array_hash(valid) == manifest['representations'][name]['valid_sha256'], 'Tensor/mask fingerprint changed.')
        bundles[name] = {'raw_X': X, 'valid': valid, 'representation': name}
    return metadata, features, bundles, manifest


def fit_temporal_preprocessor(X, valid, rows):
    observed = X[rows][valid[rows]]
    require(len(observed) > 0, 'No observed TRAIN timesteps; cannot fit preprocessing.')
    require(np.isfinite(observed).all(), 'Unresolved missingness cannot be imputed as reconstructed history.')
    state = fit_preprocessor(observed)
    return state


def transform_temporal(X, valid, state):
    values, _ = transform_features(X.reshape(-1, X.shape[-1]), state)
    values = values.reshape(X.shape)
    values[~valid] = 0
    require(np.isfinite(values).all(), 'Nonfinite Transformer input.')
    return values


def split_statistics(metadata):
    out = []
    sets = {name: set(metadata.loc[metadata.SPLIT.eq(name), 'PATIENT_ID']) for name in ('train', 'validation', 'test')}
    expected = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
    for name, ids in sets.items():
        part = metadata.loc[metadata.SPLIT.eq(name)]
        require((len(part), len(ids), int(part.RESP.sum())) == expected[name], 'Original split counts changed: ' + name)
        out.append({'SPLIT': name, 'PATIENTS': len(ids), 'SNAPSHOTS': len(part), 'RESP_0': int(part.RESP.eq(0).sum()),
                    'RESP_1': int(part.RESP.sum()), 'POSITIVE_RATE': float(part.RESP.mean())})
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        require(not sets[a].intersection(sets[b]), 'Patient overlap: ' + a + '/' + b)
    return pd.DataFrame(out)


def load_experiment():
    metadata, features, bundles, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs['audit.json'])
    states = json.loads(blobs['preprocessing.json'])
    frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT', 'SPLIT_CONFIG').toPandas()
    reference = json.loads(read_artifacts(REFERENCE_MODEL_TABLE, REFERENCE_NAMES)['training_summary.json'])
    metadata, hashes = bind_split(metadata, frozen, reference)
    split_statistics(metadata)
    require(hashes == audit['reference_hashes'] and digest_json(manifest) == audit['manifest_sha256'], 'Saved split audit changed.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
    require(records == json.loads(blobs['split.json']), 'Saved assignments changed.')
    require(digest_json(states) == audit['preprocessing_sha256'], 'Saved preprocessing changed.')
    indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
    experiments = {}
    for name, b in bundles.items():
        state = states[name]
        require(state == fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train']), 'Preprocessing does not reproduce TRAIN-only fit.')
        X = transform_temporal(b['raw_X'], b['valid'], state)
        experiments[name] = dict(b, X=X, y=metadata.RESP.to_numpy(dtype=np.float32), metadata=metadata,
            indices=indices, features=features, preprocessor=state,
            hashes={'manifest_sha256': digest_json(manifest), 'snapshot_manifest_sha256': hashes['snapshot_manifest_sha256'],
                    'preprocessing_sha256': digest_json(state), 'model_input_sha256': array_hash(X), 'mask_sha256': array_hash(b['valid'])})
    require(np.array_equal(experiments['MONTHLY']['y'], experiments['QUARTERLY']['y']), 'Representation labels differ.')
    return experiments, manifest, audit

def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names in MODEL_TYPE.FEATURES.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value

def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'

def normalize_metadata(frame):
    out = frame[["PATIENT_ID", "END_DT", "RESP"]].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Missing snapshot keys or labels.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("Patient IDs must remain nonempty strings.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("Snapshot cutoffs must be exact dates.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("Nonbinary labels.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys; no automatic deduplication is permitted.")
    return out

def align_features(snapshots, model_data, features):
    features = parse_features(features)
    expected = normalize_metadata(snapshots).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    actual = normalize_metadata(model_data)
    missing = set(features).difference(model_data.columns)
    if missing:
        raise ValueError("Selected columns absent from MODEL_DATA: " + repr(sorted(missing)))
    source = actual.copy()
    for name in features:
        # Decimal fractions are converted to float, never through an integer cast.
        source[name] = pd.to_numeric(model_data[name], errors="raise").to_numpy(dtype=np.float64)
    aligned = expected.merge(source, on=["PATIENT_ID", "END_DT"], how="left",
                             validate="one_to_one", suffixes=("", "_SOURCE"), indicator=True)
    if not aligned._merge.eq("both").all() or not aligned.RESP.eq(aligned.RESP_SOURCE).all():
        raise ValueError("Missing source keys or conflicting labels in MODEL_DATA.")
    X = aligned[features].to_numpy(dtype=np.float64)
    if np.isinf(X).any():
        raise ValueError("Infinite source feature values.")
    return expected, X

def bind_split(metadata, frozen, reference):
    original = normalize_metadata(metadata).sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized["SPLIT"] = frozen.SPLIT.to_numpy()
    normalized["SPLIT_CONFIG"] = frozen.SPLIT_CONFIG.to_numpy()
    normalized = normalized.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)
    if not original.equals(normalized[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Saved split differs from the prepared snapshots/labels.")
    if normalized[["SPLIT", "SPLIT_CONFIG"]].isna().any().any():
        raise ValueError("Incomplete frozen split.")
    if set(normalized.SPLIT) != {"train", "validation", "test"}:
        raise ValueError("Unexpected split names.")
    if normalized.groupby("PATIENT_ID").SPLIT.nunique().gt(1).any():
        raise ValueError("Patient leakage between splits.")
    if normalized.SPLIT_CONFIG.nunique() != 1:
        raise ValueError("Inconsistent split configuration.")
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {"snapshot_manifest_sha256": digest_json(records),
              "split_config_sha256": digest_json(json.loads(normalized.SPLIT_CONFIG.iloc[0]))}
    if reference.get("run_id") != "RUN_001" or reference.get("training_complete") is not True:
        raise ValueError("Expected completed original RUN_001 reference.")
    if any(reference.get("input_hashes", {}).get(k) != v for k, v in hashes.items()):
        raise ValueError("Patient assignments differ from original RUN_001 fingerprints.")
    for _, part in normalized.groupby("SPLIT"):
        if set(part.RESP) != {0, 1}:
            raise ValueError("Each split needs both outcome classes.")
    return normalized, hashes

def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}

def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)
def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())

def read_query(query):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load())

def configured_features():
    rows = read_table(SOURCE_PREFIX + "_MODEL_TYPE").select("MODEL_TYPE", "FEATURES").collect()
    if len(rows) != 1:
        raise ValueError("Expected exactly one MODEL_TYPE configuration row.")
    features = parse_features(rows[0]["FEATURES"])
    summary = read_table(SOURCE_PREFIX + "_FINAL_MODEL").toPandas()
    if "FEATURES" not in summary.columns:
        raise ValueError("FINAL_MODEL lacks FEATURES.")
    summarized = summary.FEATURES.tolist()
    if any(not isinstance(f, str) or not f for f in summarized):
        raise ValueError("Invalid FINAL_MODEL feature name.")
    comparison = {"configured_model_type": str(rows[0]["MODEL_TYPE"]),
                  "configured_feature_count": len(features), "final_summary_rows": len(summary),
                  "configured_not_in_summary": sorted(set(features) - set(summarized)),
                  "summary_not_in_configuration": sorted(set(summarized) - set(features)),
                  "summary_duplicate_names": sorted(summary.loc[summary.FEATURES.duplicated(), "FEATURES"].unique().tolist()),
                  "rule": "MODEL_TYPE.FEATURES is authoritative, in its stored order; FINAL_MODEL is an audit."}
    columns = set(read_table(SOURCE_PREFIX + "_MODEL_DATA").columns)
    missing = set(["PATIENT_ID", "END_DT", "RESP"] + features).difference(columns)
    if missing:
        raise ValueError("MODEL_DATA is missing required columns: " + repr(sorted(missing)))
    return features, comparison

def fetch_source_features(features):
    fields = ["PATIENT_ID", "END_DT", "RESP"] + features
    selected = ", ".join("M." + quote_identifier(f) for f in fields)
    # Select only the frozen cohort. Duplicated source keys remain visible and fail validation.
    query = (f"SELECT {selected} FROM {DATABASE}.DS_ML.{SOURCE_PREFIX}_MODEL_DATA M "
             f"INNER JOIN (SELECT DISTINCT PATIENT_ID, END_DT FROM {DATABASE}.DS_ML.{PREFIX}_SNAPSHOTS) S "
             "ON M.PATIENT_ID = S.PATIENT_ID AND M.END_DT = S.END_DT")
    return read_query(query).toPandas()

def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")
import base64
def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)

def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows

def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result

def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)

def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]


In [ ]:
# Read authoritative features and original snapshot population
features, configuration_audit = ordered_features()
snapshots = read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas()
metadata, snapshot_X = align_features(snapshots, fetch_source_features(features), features)
population_check(metadata)
print('Confirmed authoritative feature count:', len(features))
print('Snapshot feature matrix:', snapshot_X.shape)
display(pd.DataFrame([configuration_audit]))
for label in (0, 1):
    print('Original population examples: RESP =', label)
    display(metadata.loc[metadata.RESP.eq(label)].head(5))
display(pd.DataFrame({'FEATURE_ORDER': range(49), 'FEATURE_NAME': features,
    'SNAPSHOT_MISSING_PERCENT': np.isnan(snapshot_X).mean(axis=0) * 100,
    'SNAPSHOT_ZERO_PERCENT_ALL_ROWS': (snapshot_X == 0).mean(axis=0) * 100}))
print('Snapshot missingness and feature zeros are distinct from unavailable historical timesteps.')


### Historical calculation and coverage requirements
`HISTORICAL_RULES` is deliberately empty: supplying invented clinical formulas would change established business rules. Add verified V63 calculation functions **inside this notebook**, then register each authoritative feature with:

- `status`: EXACT or APPROXIMATED; `source`, `logic`, `evidence`, `observation_logic`, `notes`, and `builder`; APPROXIMATED also requires `approximation`.
- `builder(request, representation)` receives PATIENT_ID, original END_DT, TIME_STEP, PERIOD_START, PERIOD_END; it never receives RESP or split assignments. Return exactly one row for every requested key with VALUE, IS_OBSERVED, MAX_EVENT_DATE, MAX_AVAILABLE_DATE, OBSERVATION_EVIDENCE, PROVENANCE_KIND, PROVENANCE_NOTE.
- Recompute each feature as of PERIOD_END, using its actual rolling/calendar/day window, numerator/denominator policy, visit order, lifecycle statuses, mappings and exclusions. For period-defined measures use PERIOD_START as defined by verified business logic. Quarterly builders receive their own three-month boundaries and must recalculate; they do not receive monthly feature values to sum.
- IS_OBSERVED must come from verified source coverage **over the entire lookback needed by that feature**, including gaps, source extract availability and late-arriving information. It cannot be inferred from first/last claim or nonzero values. Event and availability maxima must describe all source rows used; null maxima are permitted for a verified observed empty window or verified static calculation.
- PROVENANCE_KIND is EVENT_DERIVED (both date maxima required), OBSERVED_EMPTY (business-defined zero), STATIC (requires `static_feature=True` and `static_rationale` in the verified rule), or UNAVAILABLE (null value). PROVENANCE_NOTE explains the evidence; these declarations are not substitutes for reviewing the source calculation.
- An observed empty window may return zero only when its business definition says zero; undefined ratios and incomplete history must use an explicit approved rule. Unavailable feature history returns null VALUE and IS_OBSERVED=0 with an explanation. Missing calculation code is a global blocker, not patient-level padding.
- A timestep is valid only if all 49 values have evidenced historical availability; partial availability is shown with AVAILABLE_FEATURE_COUNT and masks the entire token. Raw displays retain nulls; model padding is zero. All-padded snapshots stay in the cohort and bypass attention to the unchanged head with a zero pooled vector.

These checks reject out-of-window provenance, but cannot prove a supplied formula is clinically correct: review the actual V63 SQL, especially source availability and window boundaries. No approximation is pre-approved or silently supplied by these notebooks.


In [ ]:
# Dictionary transcription and explicit historical calculation registry
BUSINESS_DICTIONARY_REFERENCE = [
    {'seq': 1, 'name': 'MAX_AT_RX_NTILE_NOZOLP', 'definition': "Max # of AT Rx's from recent HCPs, excluding ambien/zolp", 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'NTILE name versus maximum-count description needs clarification; recent-HCP window, provider metric construction and as-of vintage unavailable.'},
    {'seq': 2, 'name': 'CPT_95805_SLEEP_STUDY_MULTIPLE_TRIALS', 'definition': '1 or more MSLT or MWT', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Snapshot meaning supplied; historical code set, window, source availability and V63 build SQL not verified.'},
    {'seq': 3, 'name': 'AGE', 'definition': 'Age at index date', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Historical age requires verified date-of-birth precision or an approved age-at-date calculation; do not copy snapshot age to earlier timesteps as though historical.'},
    {'seq': 4, 'name': 'TIMES_GENERIC_MIX_ADJUSTED', 'definition': 'Adjustments to generic mix', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Adjustment event, combination, overlap and counting rules are absent; historical treatment changes cannot be inferred from the name.'},
    {'seq': 5, 'name': 'UNIQUE_GENERICS_TRIED', 'definition': 'Unique generics tried', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Earlier unverified SQL counts distinct brand_name rather than generic ingredient; exact V63 normalization and history window need confirmation.'},
    {'seq': 6, 'name': 'VA_CLASS1_CN809_CNS_STIMULANTS_OTHER', 'definition': 'CNS Stimulants', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Historical drug mapping, qualifying claim status and lookback need verified V63 source logic.'},
    {'seq': 7, 'name': 'AVG_AT_PTS_NOZOLP', 'definition': 'Average # of AT patients from recent HCPs, excluding ambien/zolp', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Dictionary type Binary conflicts with average-count definition; preserve numeric source values, do not cast to a flag. Provider population, averaging and recent-HCP window missing.'},
    {'seq': 8, 'name': 'L12M_NARCO_CLAIMS', 'definition': 'Number of NT1/NT2/IH diagnosis claims in last 12 months', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Retain 12-month window at each historical cutoff; exact diagnosis set, claim-versus-service-day grain and cutoff inclusivity need V63 SQL.'},
    {'seq': 9, 'name': 'NUM_DISCONTINUATIONS', 'definition': 'Number of generic discontinuations', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Discontinuation requires the actual gap/supply/grace-period rule and rules for information known at cutoff; no such verified rule is available.'},
    {'seq': 10, 'name': 'NARCO_CLAIMS_RECENT_RATIO', 'definition': 'Percent of NT1/NT2/IH diagnosis claims in past 3 months relative to last 12 months', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Dictionary Binary conflicts with ratio definition. Need exact numerator/denominator grain, windows and zero-denominator policy; never sum ratios across months.'},
    {'seq': 11, 'name': 'DX_G47411_NARCOLEPSY_WITH_CATAPLEXY', 'definition': 'Whether they have an NT1 diagnosis', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Snapshot diagnosis flag; historical qualifying code rules and window must come from verified V63 logic.'},
    {'seq': 12, 'name': 'ATC_1_OTHER_ANTIDEPRESSANTS', 'definition': 'Whether they have other antidepressant treatments', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Need historical ATC mapping, qualifying pharmacy status and lookback; current snapshot presence cannot be replicated into past months.'},
    {'seq': 13, 'name': 'NUM_SPECIALISTS', 'definition': 'Number of unique specialist doctors they have seen', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Distinct provider count, not additive visit counts; approved specialty set and historical observation window require source SQL.'},
    {'seq': 14, 'name': 'NUM_EXCESSIVE_DAYTIME_SLEEPINESS_CLAIMS_L12M', 'definition': 'Number of excessive daytime sleepiness diagnoses', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Feature name specifies L12M; dictionary description omits the window. Verify diagnosis mapping, event grain and the exact 12-month boundary.'},
    {'seq': 15, 'name': 'CPT_99204_NEW_PATIENT_OFFICE_OR_OTHER_OUTPATIENT', 'definition': 'Number of new patient office visits (49-59 minutes)', 'type_label': 'Numeric', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Feature name is clipped at the column boundary; use this prefix only for explicit unique-match audit against authoritative names. Description time range transcribed as shown; verify approved procedure definition and lookback.'},
    {'seq': 16, 'name': 'MAX_AT_RX_NTILE_WITHZOLP', 'definition': "Max # of AT Rx's from recent HCPs, including ambien/zolp", 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'NTILE name versus maximum-count description needs clarification; provider aggregation, quantile population and historical as-of construction are unverified.'},
    {'seq': 17, 'name': 'HCPS_RX_HCP_S1_PHYSICIAN_ASSISTANT', 'definition': 'Rx received from a physician assistant', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Requires verified historical prescriber-specialty assignment, claim eligibility and lookback.'},
    {'seq': 18, 'name': 'NUM_PSYCH_COMORBIDITIES_L12M', 'definition': 'Number of Psych Comorbidities', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Dictionary Binary conflicts with number/count definition; need diagnosis family, deduplication, window and actual V63 encoding.'},
    {'seq': 19, 'name': '_G47411_G47411_', 'definition': 'Presence of NT1 with cataplexy claim followed by another in short period (within 5 visits of each other)', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Requires visit grain, chronological ordering, same-date ties, overlap and exact interpretation of within 5 visits; no verified V63 sequence SQL.'},
    {'seq': 20, 'name': 'SLEEP_MED_VISIT_RECENCY_PCT', 'definition': 'Percentage of visits to sleep medicine specialist in recent 3 months', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Dictionary Binary conflicts with percentage definition. Denominator and reference lookback are not specified; RECENCY name is not enough to infer recency-days logic.'},
    {'seq': 21, 'name': 'NUM_GAPS_30_PLUS_DAYS', 'definition': 'Number of gaps in generic usage over 30 days', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Need days-supply, overlap, stockpiling, gap boundary and trailing-gap rules; never use a future fill to establish a gap at an earlier cutoff.'},
    {'seq': 22, 'name': 'NUM_CATAPLEXY_CLAIMS_L3M', 'definition': 'Number of cataplexy dx claims in recent 3 months', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Dictionary Binary conflicts with count definition; exact V63 numeric/flag encoding, diagnosis set and date window need verification.'},
    {'seq': 23, 'name': 'CLAIMS_RX_REJECTED', 'definition': 'Number of rejected claims for Rx', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Dictionary Binary conflicts with count definition; require transaction-versus-claim grain, lifecycle status availability, reversals and lookback.'},
    {'seq': 24, 'name': 'NUM_HYPERSOMNIA_CLAIMS_L3M', 'definition': 'Number of hypersomnia diagnoses in recent 3 months', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Retain 3-month window; verify calendar-month versus fixed-day boundaries, qualifying codes and event grain.'},
    {'seq': 25, 'name': 'L6M_NARCO_CLAIMS', 'definition': 'Presence of Narcolepsy dx in recent 6 months', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Dictionary defines presence, not an additive count despite CLAIMS name. Verify V63 encoding and retain 6-month window.'},
    {'seq': 26, 'name': 'RX_INSURANCE_SEGMENT_INTEGRATED', 'definition': 'Presence of Rx paid by integrated insurance segment', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require historical paid status, insurance-segment mapping and lookback.'},
    {'seq': 27, 'name': 'RX_INSURANCE_SEGMENT_PBM', 'definition': 'Presence of Rx paid by PBM insurance segment', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require historical paid status, PBM segment mapping and lookback.'},
    {'seq': 28, 'name': 'VA_CLASS1_CN801_AMPHETAMINES', 'definition': 'Presence of Amphetamine Rx', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require approved NDC/drug mapping, historical qualifying claim status and lookback.'},
    {'seq': 29, 'name': 'PL_22', 'definition': 'Presence of on Campus Outpatient Hospital visit', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Dictionary identifies place-of-service flag; confirm V63 field mapping and historical window.'},
    {'seq': 30, 'name': '_G4710_G47411_', 'definition': 'Presence of unspecified hypersomnia diagnosis followed by Narcolepsy with cataplexy diagnosis (within 5 visits of each', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Description is clipped after each. Requires exact visit-sequence ordering/tie/grain logic and cutoff handling; do not invent the sequence calculation.'},
    {'seq': 31, 'name': 'LEVEL_2_CPT_OFFICE_OUTPATIENT_SERVICES_RECENT_PC', 'definition': 'Presence of Outpatient Service claim', 'type_label': 'Binary', 'name_complete': False, 'default_status': 'UNRESOLVED', 'notes': 'Name clipped; visible RECENT_PC prefix suggests a recent percentage but definition says presence. Match only an unambiguous authoritative prefix and verify actual V63 calculation.'},
    {'seq': 32, 'name': 'RX_PAYER_MIX_DISCOUNT_CARD', 'definition': 'Presence of Rx being paid in part with discount card', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require payer-mix mapping, qualifying lifecycle status and historical lookback.'},
    {'seq': 33, 'name': 'VISIT_TYP_VST_TELEHEALTH', 'definition': 'Presence of telehealth call', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require visit-type mapping and historical window.'},
    {'seq': 34, 'name': 'DX_SUBCAT_SPRAINS_AND_STRAINS_INITIAL_ENCOUNTER', 'definition': 'Sprains and strains, initial encounter', 'type_label': 'Binary', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Name reaches the column boundary; conservatively treat visible name as a prefix until reconciled. Verify diagnosis subcategory mapping and historical window.'},
    {'seq': 35, 'name': 'CPT_99203_NEW_PATIENT_OFFICE_OR_OTHER_OUTPATIENT', 'definition': 'Number of new patient office visits (30+ minutes) with low level of medical decision making', 'type_label': 'Numeric', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Feature name clipped; reconcile exact V63 name. Verify qualifying procedure-event grain and lookback.'},
    {'seq': 36, 'name': 'NUM_HYPERSOMNIA_CLAIMS_L12M', 'definition': 'Number of hypersomnia claims in recent 12 months', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Retain 12-month window relative to each cutoff; verify code set, date boundaries and count grain.'},
    {'seq': 37, 'name': 'VA_CLASS1_CN309_SEDATIVES_HYPNOTICS_OTHER', 'definition': 'Presence of sedatives or other hypnotic medications', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require approved historical drug mapping, qualifying pharmacy status and lookback.'},
    {'seq': 38, 'name': 'CPT_87880_DETECTION_TEST_BY_IMMUNOASSAY_WITH_DI', 'definition': 'Presence of rapid antigen detection test (RADT) for Group A Streptococcus', 'type_label': 'Binary', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Feature name clipped after DI; do not infer unseen suffix. Historical source, code handling and lookback require V63 verification.'},
    {'seq': 39, 'name': 'CPT_90791_PSYCHIATRIC_DIAGNOSTIC_EVALUATION', 'definition': 'Number of psychiatric diagnostic evaluations', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require verified procedure count grain, duplication handling and historical window.'},
    {'seq': 40, 'name': '_99213_99214_G47411_', 'definition': 'Presence of claim for established patient low medical decision making visit, followed by claim for established patient moderate medical decision making visit, followed by narcolepsy with cataplexy diagnosis (within 5 visits of each other)', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Three-event sequence requires approved visit grain, ordering, same-date tie handling and within-5-visits rule; no verified V63 historical sequence SQL.'},
    {'seq': 41, 'name': 'LEVEL_1_CPT_INFECTIOUS_AGENT_DETECTION_BY_DNA_R', 'definition': 'Presence of test DNA/RNA infectious agent test', 'type_label': 'Binary', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Feature name clipped after DNA_R; exact CPT category mapping and historical window require V63 verification.'},
    {'seq': 42, 'name': 'NUM_NEURO_SPECIALISTS', 'definition': 'Number of unique neurology specialists seen', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Distinct-provider count cannot be blindly summed over months. Require specialty mapping, provider identities and historical lookback.'},
    {'seq': 43, 'name': 'CPT_90471_ADMINISTRATION_OF_VACCINE', 'definition': 'Number of vaccines administered (non-COVID)', 'type_label': 'Numeric', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Need approved procedure count grain, non-COVID exclusion and historical lookback.'},
    {'seq': 44, 'name': '_G4733_G47411_', 'definition': 'Presence of Obstructive Sleep Apnea diagnosis followed by Narcolepsy with cataplexy diagnosis (within 5 visits of each', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'Description clipped after each. Requires approved visit sequence grain, ordering/tie rule and cutoff handling.'},
    {'seq': 45, 'name': 'DX_SUBCAT_OTHER_SPECIFIED_UPPER_RESPIRATORY_INF', 'definition': 'Presence of other upper respiratory infection diagnosis', 'type_label': 'Binary', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Name clipped after INF; match only a unique authoritative prefix and verify historical diagnosis mapping/window.'},
    {'seq': 46, 'name': 'ATC_1_SELECTIVE_SEROTONIN_REUPTAKE_INHIBITORS', 'definition': 'Presence of Rx for selective serotonin reuptake inhibitors', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require approved drug mapping, qualifying pharmacy status and historical window.'},
    {'seq': 47, 'name': 'CLAIMS_RX_REVERSED_RECENT_PCT', 'definition': 'Presence of reversed Rx claim', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'UNRESOLVED', 'notes': 'RECENT_PCT name suggests a percentage while definition/type say presence. Need V63 numerator, denominator or flag rule, status availability and time window.'},
    {'seq': 48, 'name': 'CPT_99395_ESTABLISHED_PATIENT_PERIODIC_PREVENTIV', 'definition': 'Number of Established patient periodic preventive medicine examinations (ages 18-39)', 'type_label': 'Numeric', 'name_complete': False, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Feature name clipped after PREVENTIV; reconcile exact V63 name and confirm procedure count grain/window.'},
    {'seq': 49, 'name': 'LEVEL_2_CPT_MOLECULAR_TESTING', 'definition': 'Presence of molecular testing procedure', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'Require approved procedure category mapping and historical lookback.'},
    {'seq': 50, 'name': 'DX_G4719_OTHER_HYPERSOMNIA', 'definition': 'Presence of other hypersomnia diagnosis', 'type_label': 'Binary', 'name_complete': True, 'default_status': 'SNAPSHOT_ONLY', 'notes': 'This is dictionary entry 50, not proof it is a 50th model predictor; only the authoritative V63 ordered array controls inclusion.'},
]

# Register only verified V63 historical builders here; do not copy snapshot values across time.
HISTORICAL_RULES = {}


In [ ]:
# Display dictionary reconciliation and all 49 reconstruction statuses
audit, status_counts, dictionary_only = reconstruction_audit(features, BUSINESS_DICTIONARY_REFERENCE, HISTORICAL_RULES)
display(audit)
display(status_counts.rename_axis('STATUS').reset_index(name='FEATURE_COUNT'))
for status, count in status_counts.items():
    print(f'{status} = {count}')
print('Status total:', int(status_counts.sum()))
print('Dictionary reference rows:', len(BUSINESS_DICTIONARY_REFERENCE), '| Authoritative model inputs:', len(features))
if not dictionary_only.empty:
    print('Dictionary rows not matched to the authoritative model list (clipped text may require confirmation):')
    display(dictionary_only)
if len(dictionary_only) == 1 and audit.MATCH.ne('UNRESOLVED').all():
    print('The reference dictionary has one entry outside the configured predictor list:', dictionary_only.iloc[0]['name'])
    print('It is not added as a 50th feature. Why it was excluded upstream is not inferred here.')
else:
    print('Do not infer the 49-versus-50 cause until unmatched/clipped names are resolved against V63.')


In [ ]:
# Construct and inspect date grids before any historical value calculation
monthly_grid = sequence_grid(metadata, 'MONTHLY')
quarterly_grid = sequence_grid(metadata, 'QUARTERLY')
verify_periods(monthly_grid, quarterly_grid)
print('TIME_STEP 0 = most recent; increasing indices = older. Current bucket ends at END_DT.')
print('Date-grid rows only; these are not reconstructed model inputs:', len(monthly_grid), len(quarterly_grid))
display(monthly_grid.head(24).rename(columns={'PERIOD_START': 'MONTH_START', 'PERIOD_END': 'MONTH_END'}))
display(quarterly_grid.head(8).rename(columns={'TIME_STEP': 'QUARTER_TIME_STEP', 'PERIOD_START': 'QUARTER_START', 'PERIOD_END': 'QUARTER_END'}))


In [ ]:
# Stop on unsupported reconstruction; build monthly first and quarterly second
require_reconstruction(audit, HISTORICAL_RULES)
bundles = {}
bundles['MONTHLY'] = construct_sequence(monthly_grid, features, audit, HISTORICAL_RULES, 'MONTHLY')
bundles['QUARTERLY'] = construct_sequence(quarterly_grid, features, audit, HISTORICAL_RULES, 'QUARTERLY')
verify_periods(bundles['MONTHLY']['long'], bundles['QUARTERLY']['long'])
print('Actual monthly tensor shape:', bundles['MONTHLY']['X'].shape)
print('Actual quarterly tensor shape:', bundles['QUARTERLY']['X'].shape)


In [ ]:
# Inspect values, padding, sparsity and patient histories
coverage_rows = []
for name in ('MONTHLY', 'QUARTERLY'):
    bundle = bundles[name]
    report, feature_sparsity, history_distribution = sparsity_report(bundle, features)
    coverage_rows.append(report)
    print(name, 'feature sparsity on observed feature values; padded zeros excluded from that denominator')
    display(feature_sparsity)
    display(history_distribution)
    display_sequence(bundle, features)
    # Model padding demonstration uses the same patient/date/step keys as the raw display.
    model_sample = bundle['long'][['PATIENT_ID', 'END_DT', 'RESP', 'TIME_STEP', 'IS_VALID_TIMESTEP', 'IS_PADDED']].head(24).copy()
    model_sample[features] = bundle['X'].reshape(-1, 49)[:len(model_sample)]
    print(name, 'raw model tensor values before TRAIN-only standardization')
    display(model_sample)
display(pd.DataFrame(coverage_rows))
print('Feature sparsity is not timestep padding; numeric zero after scaling is not a sparsity measure.')


In [ ]:
# Save verified temporal inputs privately for the same four-notebook pipeline
artifacts = prepared_blobs(metadata, features, bundles, audit, configuration_audit, snapshot_X)
save_artifacts(PREPARED_TABLE, artifacts)
print('Prepared inputs saved and verified:', PREPARED_TABLE)
print('Continue with notebook 02. No patient records, tensors or outputs belong in Git.')
